# SparkRules — Decision Tables

Decision tables let you define rules as rows in a table instead of writing DRL.
This is useful for business users who think in spreadsheets.

```bash
pip install sparkrules
```

In [ ]:
from sparkrules.model.decision_table import (
    DecisionTable, InputColumn, OutputColumn, Row,
    ColumnType, HitPolicy, evaluate_decision_table, dt_to_json,
)

## 1. Create a decision table

This table determines shipping cost based on customer tier and order weight.

In [ ]:
shipping_table = DecisionTable(
    name="shipping_cost",
    hit_policy=HitPolicy.FIRST,
    inputs=(
        InputColumn("tier", "customer_tier", ColumnType.STRING, "=="),
        InputColumn("weight", "package_weight_kg", ColumnType.INT, ">="),
    ),
    outputs=(
        OutputColumn("cost", "shipping_cost_usd", ColumnType.INT),
        OutputColumn("method", "shipping_method", ColumnType.STRING),
    ),
    rows=(
        Row(("gold", 0, 0, "express"), priority=0),       # Gold: free express
        Row(("silver", 10, 15, "standard"), priority=1),   # Silver heavy: $15
        Row(("silver", 0, 5, "standard"), priority=2),     # Silver light: $5
        Row(("bronze", 10, 25, "standard"), priority=3),   # Bronze heavy: $25
        Row(("bronze", 0, 10, "economy"), priority=4),     # Bronze light: $10
    ),
)

print(f"Table: {shipping_table.name}")
print(f"Hit policy: {shipping_table.hit_policy.name}")
print(f"Rows: {len(shipping_table.rows)}")

## 2. Evaluate facts against the table

In [ ]:
# Gold customer — free express shipping
r1 = evaluate_decision_table(shipping_table, {"customer_tier": "gold", "package_weight_kg": 5})
print(f"Gold customer: {r1}")

# Silver customer, heavy package
r2 = evaluate_decision_table(shipping_table, {"customer_tier": "silver", "package_weight_kg": 15})
print(f"Silver heavy: {r2}")

# Bronze customer, light package
r3 = evaluate_decision_table(shipping_table, {"customer_tier": "bronze", "package_weight_kg": 3})
print(f"Bronze light: {r3}")

## 3. Hit policies

| Policy | Behavior |
|--------|----------|
| `FIRST` | First matching row wins |
| `UNIQUE` | Exactly one row must match |
| `PRIORITY` | Highest priority matching row wins |
| `COLLECT` | All matching rows returned |

## 4. Export to JSON

In [ ]:
import json

table_json = dt_to_json(shipping_table)
print(json.dumps(table_json, indent=2))